# Kahneman Framing × TRIBE v2 — Quick Colab Demo

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/akifnu/DSprojects/blob/main/tribev2/notebooks/Framing_RCT_NoSetup.ipynb)

**3 classic pairs · 6 texts · text-only · no audio**

1. **Runtime → GPU** (A100 recommended)
2. Colab **Secrets** → `HF_TOKEN`
3. **Runtime → Run all** (restarts once on first install — run again if needed)

> If you see `ModuleNotFoundError`: **Runtime → Factory reset runtime**, then **Run all**.

In [ ]:
import os, shutil, subprocess, sys
from pathlib import Path

MARKER = Path('/content/.tribev2_colab_ready_v7')
REPO_DIR = Path('/content/DSprojects')
REPO_URL = 'https://github.com/akifnu/DSprojects.git'
SRC = REPO_DIR / 'tribev2/src'
COLAB_PY = SRC / 'tribe_capabilities/colab.py'

def ensure_repo():
    if REPO_DIR.exists() and not COLAB_PY.exists():
        shutil.rmtree(REPO_DIR, ignore_errors=True)
    if not REPO_DIR.exists():
        subprocess.check_call(['git', 'clone', '--depth', '1', REPO_URL, str(REPO_DIR)])
    else:
        subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only', 'origin', 'main'], check=False)
    if not COLAB_PY.exists():
        raise RuntimeError('colab.py missing after clone — open notebook from GitHub main branch link above.')

if not MARKER.exists():
    ensure_repo()
    req = REPO_DIR / 'tribev2/requirements-colab.txt'
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(req)])
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO_DIR / 'tribev2')])
    MARKER.write_text('ok')
    import IPython
    IPython.get_ipython().kernel.do_shutdown(restart=True)
else:
    ensure_repo()
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO_DIR / 'tribev2')])

sys.path.insert(0, str(SRC))
print('Dependencies ready:', COLAB_PY.exists())

In [ ]:
import sys
from pathlib import Path

REPO_DIR = Path('/content/DSprojects')
sys.path.insert(0, str(REPO_DIR / 'tribev2/src'))

import pandas as pd
from scipy import stats

# Fallback: fetch colab.py from GitHub if local clone is stale
try:
    from tribe_capabilities.colab import run_quick_demo
except ModuleNotFoundError:
    import urllib.request
    base = 'https://raw.githubusercontent.com/akifnu/DSprojects/main/tribev2/src/tribe_capabilities'
    pkg = REPO_DIR / 'tribev2/src/tribe_capabilities'
    pkg.mkdir(parents=True, exist_ok=True)
    for name in ('colab.py', 'inference.py', 'config.py', 'environment.py'):
        dest = pkg / name
        dest.write_bytes(urllib.request.urlopen(f'{base}/{name}').read())
    import importlib
    import tribe_capabilities.colab as colab_mod
    importlib.reload(colab_mod)
    from tribe_capabilities.colab import run_quick_demo

demo = run_quick_demo()

if not demo.rows:
    raise RuntimeError('No completed pairs. Restart runtime and Run all. Errors: ' + str(demo.errors))

df = pd.DataFrame(demo.rows)
display(df[['id', 'domain', 'gain_mean_abs', 'loss_mean_abs', 'loss_minus_gain']])

gain_vals = df['gain_mean_abs'].values
loss_vals = df['loss_mean_abs'].values
diff = loss_vals - gain_vals
_, p_val = stats.ttest_rel(loss_vals, gain_vals)
cohens_dz = diff.mean() / diff.std(ddof=1) if diff.std(ddof=1) > 0 else float('nan')

print('\n--- Kahneman framing (3-pair quick demo) ---')
print(f'Pairs: {len(df)} | Loss > gain: {(diff > 0).sum()}/{len(df)}')
print(f'Mean diff (loss - gain): {diff.mean():.4f} | p = {p_val:.4f} | dz = {cohens_dz:.3f}')
print(f'Checkpoint: {demo.checkpoint_path}')
if demo.errors:
    print('Non-fatal errors:', demo.errors)